### **Project Title: Space-Time Clustering of Mobility Patterns and Air Quality Hotspots**

Install Required Packages
and import Libraries

In [ ]:
!pip install pygeohash
!pip install folium
!pip install uszipcode
!pip installgeopandas



ERROR: unknown command "installgeopandas" - maybe you meant "install"
  Preparing metadata (setup.py) ... done
  Created wheel for geohash: filename=Geohash-1.0-py3-none-any.whl size=15521 sha256=b81bad9d7194e3efb93d7482d582cdea6105f02114cbd3474267bbc966cef48b
  Stored in directory: /root/.cache/pip/wheels/b8/13/56/22a1fc3c0d613735c1709eb859a70a99f2877c6fefe0fc364b
Successfully built geohash


In [ ]:
from datascience import *
import pandas as pd
import geopandas as gpd
import pygeohash as gh
import numpy as np
from shapely.geometry import Polygon
#from shapely.geometry import Point
from geopandas.tools import sjoin
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')


## **1. Data Exploration & Cleaning**

Steps:
Inspect each dataset:

*   Check timestamp formats, spatial coordinates, and column consistency.

*   Remove erroneous coordinates like (0,0).

*   Convert timestamps to a consistent format.

*   Clean and handle missing/null values.




Read CSV files NYC_AQ, NYC_pm,nyc1 and gejson file nyc_polygon

In [ ]:

#Read the CSV file containing PM sensors readings and AQ file
PM_data = pd.read_csv('https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-space-time-clustering-for-aq/refs/heads/main/Datasets/NYC_PM.csv',index_col=False)
AQ_data = pd.read_csv('https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-space-time-clustering-for-aq/refs/heads/main/Datasets/NYC_AQ.csv',index_col=False)

#nyc1_file ="--------------"
#nyc1_data = pd.read_csv(nyc1_file)

#Read the GeoJSON file containing neighborhood boundaries into a GeoDataFrame
nyc_neighborhoods = gpd.read_file('https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-space-time-clustering-for-aq/refs/heads/main/Datasets/nyc_polygon.geojson')

In [ ]:
print("AQ_data",AQ_data.shape)
print("PM_data",PM_data.shape)

#print("nyc1_data",nyc1_data.shape)

print("nyc_neighborhoods",nyc_neighborhoods.shape)

AQ_data (169999, 31)
PM_data (118765, 33)
nyc_neighborhoods (310, 5)


In [ ]:
AQ_data.head(5)

,SensorID,time,latitude,longitude,bin0,bin1,bin2,bin3,bin4,bin5,...,bin17,bin18,bin19,bin20,bin21,bin22,bin23,temperature,humidity,pm25
0,NYCP2_CS01A,1631277304,40.847672,-73.869316,11,1,1,0,0,0,...,0,0,0,0,0,0,0,23.7,57.3,4.508813
1,NYCP2_CS01A,1631277308,40.847668,-73.869316,22,4,1,0,0,2,...,0,0,0,0,0,0,0,23.7,57.8,5.462420
2,NYCP2_CS01A,1631277313,40.847649,-73.869362,40,1,1,0,0,1,...,0,0,0,0,0,0,0,23.7,57.8,5.154881
3,NYCP2_CS01A,1631277318,40.847649,-73.869362,26,1,0,0,0,0,...,0,0,0,0,0,0,0,23.6,57.6,4.508813
4,NYCP2_CS01A,1631277323,40.847649,-73.869362,44,4,0,1,0,0,...,0,0,0,0,0,0,0,23.6,57.5,5.539503


In [ ]:
PM_data.head(5)

,SensorID,time,latitude,longitude,bin0,bin1,bin2,bin3,bin4,bin5,...,bin19,bin20,bin21,bin22,bin23,temperature,humidity,pm1,pm25,pm10
0,NYCP1_01A,1579618560,40.847183,-73.870087,23,1,2,0,0,0,...,0.0,0.0,0.0,0.0,0.0,16.3,15.2,1.44,5.91,11.35
1,NYCP1_01A,1579618560,40.847183,-73.870094,18,2,1,0,0,0,...,0.0,0.0,0.0,0.0,0.0,16.2,15.1,1.05,1.18,1.18
2,NYCP1_01A,1579618560,40.847179,-73.870094,18,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,16.1,15.1,0.74,0.76,0.76
3,NYCP1_01A,1579618560,40.847179,-73.870094,18,1,2,0,0,0,...,0.0,0.0,0.0,0.0,0.0,16.1,15.2,1.15,4.48,47.36
4,NYCP1_01A,1579618560,40.847179,-73.870094,20,3,0,2,2,0,...,0.0,0.0,0.0,0.0,0.0,16.0,15.2,2.13,5.77,6.18


In [ ]:


#nyc1_data.head(5)



In [ ]:
nyc_neighborhoods.head(5)

,neighborhood,boroughCode,borough,@id,geometry
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702..."
1,Alley Pond Park,4,Queens,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.74333 40.73888, -73.74371 40.739..."
2,Arden Heights,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.16983 40.56108, -74.16982 40.561..."
3,Arlington,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.15975 40.64142, -74.15998 40.641..."
4,Arrochar,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.06078 40.59319, -74.06079 40.593..."


In [ ]:
#remove erroneous coordinates (0,0)
#Convert the Unix epoch time column to datetime and set it as a new column
#######AQ_data######
AQ_data = \
AQ_data[(AQ_data ['latitude']!=0) & \
       (AQ_data ['longitude'] !=0)]
AQ_data ['datetime'] = pd.to_datetime(AQ_data['time'], unit='s')
print("AQ_data",AQ_data.shape)
#######PM_data######
PM_data = \
PM_data[(PM_data ['latitude']!=0) & \
       (PM_data ['longitude'] !=0)]
PM_data ['datetime'] = pd.to_datetime(PM_data['time'], unit='s')
print("PM_data",PM_data.shape)



#######nyc1_data######

# both steps should be done here.... clean and unify datetime format

#nyc1_data ['Pickup_datetime'] = pd.to_datetime(nyc1_data['lpep_pickup_datetime'], unit='s')
#nyc1_data ['Dropoff_datetime'] = pd.to_datetime(nyc1_data['Lpep_dropoff_datetime'], unit='s')

#print("nyc1_data",nyc1_data.shape)


AQ_data (169999, 32)
PM_data (118765, 34)


## **2. Data Integration**

Aggregate all datasets into a common spatial grid and temporal resolution.

Spatial Aggregation:

*    Create geohash cells over NYC.
*    Spatially join GPS/mobility data and AQ data to these grid cells.

Temporal Aggregation:
*    Resample time to a fixed interval.
*    For each grid cell & time slice, calculate:

      *    Mobility: Count of trips, speed, or density.

      *    Air Quality: Mean PM2.5, humidity, and temprature.



🗺 Spatial Grid:

In [ ]:
#Set configuration
geohash_precision= 6

In [ ]:
#Generate Geohash for each tuple (long,lat)
AQ_data['geohash']=AQ_data.apply(lambda x: gh.encode(x.latitude,x.longitude,precision=geohash_precision),axis=1)
PM_data['geohash']=PM_data.apply(lambda x: gh.encode(x.latitude,x.longitude,precision=geohash_precision),axis=1)

In [ ]:
# generate geohashes for pickup and dropoff points for nyc1.csv file

#nyc1_data['Pickup_geohash'] = nyc1_data.apply(lambda x: gh.encode(x['Pickup_latitude'], x['Pickup_longitude'], precision=geohash_precision), axis=1)
#nyc1_data['Dropoff_geohash'] = nyc1_data.apply(lambda x: gh.encode(x['Dropoff_latitude'], x['Dropoff_longitude'], precision=geohash_precision), axis=1)


⏱ Temporal Resolution:
* Round timestamps to 20min for alignment.

🧮 Aggregation:
For each spatial cell (geohash) and time interval (20min):
* Count number of trips, using the total pickup and dropoff points for each (geohash, time_bin).
* Calculate mean PM2.5 , humidity, and temperature.

In [ ]:
#######AQ_data#######
# Round the datetime to 20-minute intervals
AQ_data['time_bin'] = AQ_data['datetime'].dt.floor('20min')

# Group by geohash and the time_bin column
AQ_aggregated = AQ_data.groupby(['geohash', 'time_bin']).agg({
    'temperature': 'mean',
    'humidity': 'mean',
    'pm25': 'mean',
    # Add any other columns if you'd like to aggregate
}).reset_index()
print("AQ_aggregated",AQ_aggregated.shape)
AQ_aggregated.head(5)


AQ_aggregated (4585, 5)


,geohash,time_bin,temperature,humidity,pm25
0,dr5rte,2021-10-29 14:40:00,13.700000,64.000000,3.187280
1,dr5ry2,2021-10-29 13:40:00,14.000000,59.300000,3.084182
2,dr5rz9,2021-09-22 14:20:00,29.600000,60.824242,11.387078
3,dr5rz9,2021-09-22 15:00:00,29.501471,64.905147,11.063003
4,dr5rz9,2021-09-22 15:20:00,28.208036,69.866071,9.865907


In [ ]:
#######PM_data#######
# Round the datetime to 20-minute intervals
PM_data ['time_bin'] = PM_data ['datetime'].dt.floor('20min')

# Group by geohash and the time_bin column
PM_aggregated = PM_data.groupby(['geohash', 'time_bin']).agg({
    'temperature': 'mean',
    'humidity': 'mean',
    'pm25': 'mean',
    # Add any other columns if you'd like to aggregate
}).reset_index()
print("PM_aggregated",PM_aggregated.shape)
PM_aggregated.head(5)


PM_aggregated (1635, 5)


,geohash,time_bin,temperature,humidity,pm25
0,dr57we,2020-02-05 13:40:00,5.40000,70.600000,0.280000
1,dr57we,2020-02-05 15:40:00,9.30000,46.100000,0.000000
2,dr5ref,2020-02-05 12:20:00,6.30000,68.681818,4.754545
3,dr5reg,2020-02-05 12:20:00,6.30000,69.102410,3.041928
4,dr5reg,2020-02-05 12:40:00,6.18961,69.945022,3.505498


In [ ]:
#######nyc1_data#######
# Round the datetime to 20-minute intervals
#nyc1_data ['Pickup_time_bin'] = nyc1_data ['Pickup_datetime'].dt.floor('20min')
#nyc1_data ['Dropoff_time_bin'] = nyc1_data ['Dropoff_datetime'].dt.floor('20min')

# Group by geohash and the time_bin column
# we will get number of trips starting or ending in each geohash during each 20-minute interval.
nyc1_Pickup_points_aggregated = nyc1_data.groupby(['Pickup_geohash', 'Pickup_time_bin']).agg({
    'VendorID': 'count',
    # Add any other columns if you'd like to aggregate
}).reset_index()
nyc1_Dropoff_points_aggregated = nyc1_data.groupby(['Dropoff_geohash', 'Dropoff_time_bin']).agg({
    'VendorID': 'count',
    # Add any other columns if you'd like to aggregate
}).reset_index()

print(nyc1_Pickup_points_aggregated.head(5))
print(nyc1_Dropoff_points_aggregated.head(5))



A clear summary of total mobility activity per (geohash area and 20-minute time bin).

In [ ]:
# Rename columns for consistency
nyc1_Pickup_points_aggregated = nyc1_Pickup_points_aggregated.rename(columns={
    'Pickup_geohash': 'geohash',
    'Pickup_time_bin': 'time_bin',
    'VendorID': 'pickup_trip_count'
})

nyc1_Dropoff_points_aggregated = nyc1_Dropoff_points_aggregated.rename(columns={
    'Dropoff_geohash': 'geohash',
    'Dropoff_time_bin': 'time_bin',
    'VendorID': 'dropoff_trip_count'
})

# Merge on geohash and time_bin
nyc1_joined = pd.merge(
    nyc1_Pickup_points_aggregated,
    nyc1_Dropoff_points_aggregated,
    on=['geohash', 'time_bin'],
    how='outer'
)

# Fill missing values with 0
nyc1_joined[['pickup_trip_count', 'dropoff_trip_count']] = nyc1_joined[['pickup_trip_count', 'dropoff_trip_count']].fillna(0).astype(int)

# Add total_trip_count column
nyc1_joined['total_trip_count'] = nyc1_joined['pickup_trip_count'] + nyc1_joined['dropoff_trip_count']


print(nyc1_joined.head())

Note: pd.merge(..., how='outer') — Full Outer Join
🔍 What it does:
Keeps all rows from both tables.

Fills in NaN for missing values where there is no match on the join keys.

🧠 Think of it as:
“Keep everything — even if it only exists in one side.”

Now we will join the three datasets:

nyc1_joined (trip data)
AQ_aggregated (air quality from AQ_data)
PM_aggregated (air quality from PM_data)

by merge them on ['geohash', 'time_bin'].






In [ ]:
# Step 1: Merge trip data with AQ data
merged_1 = pd.merge(
    nyc1_joined,
    AQ_aggregated,
    on=['geohash', 'time_bin'],
    how='outer',  # Keep all spatial-temporal combinations
    suffixes=('', '_aq'))

# Step 2: Merge the result with PM data
final_merged = pd.merge(
    merged_1,
    PM_aggregated,
    on=['geohash', 'time_bin'],
    how='outer',
    suffixes=('', '_pm'))

# Step 3: Fill missing counts with 0, and optionally drop/rename columns if needed

final_merged[['pickup_trip_count', 'dropoff_trip_count', 'total_trip_count']] = final_merged[[
    'pickup_trip_count', 'dropoff_trip_count', 'total_trip_count'
]].fillna(0).astype(int)

# View result
print(final_merged.head())


In [ ]:
#find mean for the AQ parameters (temprature, humidity,and pm25)

# Create new columns that store the average of AQ and PM values, handling NaNs safely
final_merged['pm25_mean'] = final_merged[['pm25', 'pm25_pm']].mean(axis=1, skipna=True)
final_merged['humidity_mean'] = final_merged[['humidity', 'humidity_pm']].mean(axis=1, skipna=True)
final_merged['temperature_mean'] = final_merged[['temperature', 'temperature_pm']].mean(axis=1, skipna=True)

final_merged = final_merged.drop(columns=[
    'pm25', 'pm25_pm',
    'humidity', 'humidity_pm',
    'temperature', 'temperature_pm'])

# View result
print(final_merged.head())
print(type(final_merged))

🗺 Spatial join:

Useing geohash_to_polygon new defined function to convert each geohash into a bounding box  (polygon).

Performs a polygon-to-polygon spatial join.


In [ ]:
#not sure about this satage

# geohash to polygon function
def geohash_to_polygon(gh_str):
    lat, lon = gh.decode(gh_str)
    lat_err, lon_err = gh.decode_exactly(gh_str)[2:]
    lat_min = lat - lat_err
    lat_max = lat + lat_err
    lon_min = lon - lon_err
    lon_max = lon + lon_err
    return Polygon([
        (lon_min, lat_min),
        (lon_max, lat_min),
        (lon_max, lat_max),
        (lon_min, lat_max),
        (lon_min, lat_min)
    ])

# 1. Convert geohash to polygon and then to Geodataframe
final_merged['geometry'] = final_merged['geohash'].apply(geohash_to_polygon)
print(type(final_merged))
gdf = gpd.GeoDataFrame(final_merged, geometry='geometry', crs='EPSG:4326')
print(type(gdf))


# 2. Perform polygon-to-polygon overlay to compute intersections
overlap = gpd.overlay(gdf, nyc_neighborhoods, how='intersection')

# 3. Add area of intersection
overlap['intersect_area'] = overlap.geometry.area



#### use area-weighted averaging for trips count/ and uniform data for pm25, temprature and humidity???


print(largest_only.shape)
print(overlap.shape)